In [1]:
import pandas as pd

In [2]:
# Određivanje putanje do CSV datoteke
CSV_FILE_PATH = "../data/atp_matches_all.csv"


In [3]:
# Učitavanje CSV datoteke
df = pd.read_csv(CSV_FILE_PATH, delimiter=',')
print("CSV size before: ", df.shape)

CSV size before:  (50179, 49)


In [4]:
# Uklanjanje Davis Cup mečeva
# Davis Cup mečevi su uklonjeni prije daljnje obrade jer imaju drugačiju strukturu podataka i velik broj nedostajućih vrijednosti.
df = df[df["tourney_level"] != "D"]

In [5]:
# Micanje crte i pretvaranje u int
df['tourney_id'] = df['tourney_id'].str.replace('-', '', regex=False).astype(int)

In [6]:
# Pretvorba tourney_date u standardni datum kako bi se omogućila jednostavnija vremenska analiza podataka.

df["tourney_date"] = pd.to_datetime(df["tourney_date"], format="%Y%m%d")


In [7]:
# Popunjavanje seed (0 = nije seedan)
df['winner_seed'] = df['winner_seed'].fillna(0)
df['loser_seed'] = df['loser_seed'].fillna(0)

In [8]:
# Popunjavanje entry (Main Draw = standardni ulaz)
df['winner_entry'] = df['winner_entry'].fillna('Main Draw')
df['loser_entry'] = df['loser_entry'].fillna('Main Draw')

In [9]:
# Godine tenisača se mijenjaju te zato nisu adekvatne
df = df.drop(columns=['winner_age', 'loser_age'])

In [10]:
# Igrač u tom trenutku nije imao rang ili bodove i stavljamo 0

df["winner_rank"] = df["winner_rank"].fillna(0)
df["loser_rank"] = df["loser_rank"].fillna(0)

df["winner_rank_points"] = df["winner_rank_points"].fillna(0)
df["loser_rank_points"] = df["loser_rank_points"].fillna(0)

In [ ]:
# Nedostajuće vrijednosti za visinu igrača popunjene su medijanom kako bi se zadržali svi zapisi.

df["winner_ht"] = df["winner_ht"].fillna(df["winner_ht"].median())
df["loser_ht"] = df["loser_ht"].fillna(df["loser_ht"].median())


In [12]:
# Učitavanje CSV mapa
tourney_map = pd.read_csv("tourney_countries.csv")
players_map = pd.read_csv("players_countries.csv")

# Dodavanje grada i države turnira
df = df.merge(tourney_map, on="tourney_name", how="left")

# IOC -> puni nazivi država igrača
ioc_dict = dict(zip(players_map["ioc"], players_map["country"]))
df["winner_country"] = df.pop("winner_ioc").map(ioc_dict)
df["loser_country"] = df.pop("loser_ioc").map(ioc_dict)

# Premještanje stupaca na željena mjesta
df.insert(df.columns.get_loc("tourney_name"), "tourney_country", df.pop("tourney_country"))
df.insert(df.columns.get_loc("tourney_name"), "tourney_city", df.pop("tourney_city"))
df.insert(df.columns.get_loc("winner_seed"), "winner_country", df.pop("winner_country"))
df.insert(df.columns.get_loc("loser_seed"), "loser_country", df.pop("loser_country"))

In [13]:
df = df.dropna()

In [14]:
print("CSV size after: ", df.shape) # Ispis broja redaka i stupaca nakon predprocesiranja
print(df.head()) # Ispis prvih redaka dataframe-a


CSV size after:  (43159, 49)
   tourney_id tourney_country tourney_city tourney_name surface  draw_size  \
0     2000717   United States      Orlando      Orlando    Clay         32   
1     2000717   United States      Orlando      Orlando    Clay         32   
2     2000717   United States      Orlando      Orlando    Clay         32   
3     2000717   United States      Orlando      Orlando    Clay         32   
4     2000717   United States      Orlando      Orlando    Clay         32   

  tourney_level tourney_date  match_num  winner_id  ... w_bpFaced  l_ace l_df  \
0             A   2000-05-01          1     102179  ...      15.0   13.0  4.0   
1             A   2000-05-01          2     103602  ...       6.0    0.0  0.0   
2             A   2000-05-01          3     103387  ...       0.0    2.0  2.0   
3             A   2000-05-01          4     101733  ...      12.0    4.0  6.0   
4             A   2000-05-01          5     101727  ...       1.0    0.0  3.0   

  l_svpt l_1stI

In [15]:
# Count if there are duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicates: {duplicates}") # Ispis broja duplikata

Number of duplicates: 0


In [16]:
# Random dijeljenje skupa podataka na dva dijela 80:20 
df20 = df.sample(frac=0.2, random_state=1)
df80 = df.drop(df20.index)
print("CSV size 80: ", df80.shape)
print("CSV size 20: ", df20.shape)

CSV size 80:  (34527, 49)
CSV size 20:  (8632, 49)


In [17]:
df80.to_csv("processed/atp_matches_processed_80.csv", index=False)
df20.to_csv("processed/atp_matches_processed_20.csv", index=False)